# 🏦 Advanced Loan Default Risk Assessment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jameskoero/loan-risk-assessment/blob/main/loan_risk_assessment.ipynb)

---

> **Objective:** Build a production-ready machine-learning pipeline that predicts the probability of loan default, handles class imbalance with SMOTE, tunes multiple classifiers, and provides business-oriented evaluation with cost-sensitive threshold optimization.

---

## 📋 Table of Contents

1. [Setup & Imports](#1)
2. [Data Generation & Loading](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Preprocessing](#4)
5. [Class Imbalance – SMOTE](#5)
6. [Model Training & Comparison](#6)
7. [Hyperparameter Tuning](#7)
8. [Model Evaluation](#8)
9. [Model Explainability (SHAP)](#9)
10. [Business-Oriented Evaluation](#10)
11. [Final Model Selection & Export](#11)
12. [Prediction Function](#12)
13. [Conclusion & Business Recommendations](#13)
14. [Requirements](#14)

## 1. Setup & Imports <a id='1'></a>

Install any missing libraries and import everything we need for the full pipeline.

In [ ]:
# Install optional dependencies (uncomment if needed in Colab)
# !pip install -q xgboost lightgbm shap optuna imbalanced-learn

import warnings, os, joblib, json
warnings.filterwarnings('ignore')

# ── Standard ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualisation ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Sklearn ───────────────────────────────────────────────────────────────────
from sklearn.model_selection import (
    train_test_split, GridSearchCV, StratifiedKFold, learning_curve
)
from sklearn.preprocessing import StandardScaler, LabelEncoder, label_binarize
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    brier_score_loss
)
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

# ── Imbalanced-learn ──────────────────────────────────────────────────────────
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# ── XGBoost / LightGBM (optional) ────────────────────────────────────────────
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost not available – skipping.")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
except ImportError:
    LIGHTGBM_AVAILABLE = False
    print("LightGBM not available – skipping.")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("SHAP not available – skipping explainability section.")

# ── Global plot settings ──────────────────────────────────────────────────────
sns.set_theme(style="darkgrid", palette="muted")
PALETTE = sns.color_palette("muted")
PLOTS_DIR = "plots"
os.makedirs(PLOTS_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("✅ All imports successful.")
print(f"   XGBoost   : {XGBOOST_AVAILABLE}")
print(f"   LightGBM  : {LIGHTGBM_AVAILABLE}")
print(f"   SHAP      : {SHAP_AVAILABLE}")


## 2. Data Generation & Loading <a id='2'></a>

We generate a **realistic synthetic loan dataset** (10 000 applicants) with features typical of credit-risk scorecards:

| Feature | Description |
|---------|-------------|
| `loan_amnt` | Requested loan amount (USD) |
| `int_rate` | Annual interest rate (%) |
| `annual_inc` | Applicant annual income (USD) |
| `dti` | Debt-to-income ratio |
| `fico_score` | FICO credit score |
| `delinq_2yrs` | Delinquencies in the past 2 years |
| `open_acc` | Open credit lines |
| `revol_util` | Revolving utilisation rate (%) |
| `total_acc` | Total credit lines ever opened |
| `home_ownership` | RENT / OWN / MORTGAGE |
| `purpose` | Loan purpose |
| `emp_length` | Employment length (years bucket) |
| `grade` | Internal loan grade (A–G) |
| **`default`** | **Target – 1 = defaulted, 0 = paid** |

> **Note:** Replace the generation block with your own `pd.read_csv(...)` call to use a real dataset (e.g., Lending Club or Home Credit).

In [ ]:
def generate_loan_dataset(n: int = 10_000, random_state: int = 42) -> pd.DataFrame:
    """Generate a synthetic loan dataset that mimics real credit-risk distributions."""
    rng = np.random.default_rng(random_state)

    fico        = rng.normal(680, 60, n).clip(300, 850)
    annual_inc  = np.exp(rng.normal(10.8, 0.6, n)).clip(15_000, 500_000)
    loan_amnt   = rng.choice([5_000, 10_000, 15_000, 20_000, 25_000, 35_000], n)
    int_rate    = rng.normal(12, 4, n).clip(5, 30)
    dti         = rng.gamma(3, 3, n).clip(0, 50)
    delinq_2yrs = rng.poisson(0.3, n)
    open_acc    = rng.integers(3, 25, n)
    revol_util  = rng.beta(2, 3, n) * 100
    total_acc   = rng.integers(5, 60, n)

    home_ownership = rng.choice(["RENT", "OWN", "MORTGAGE"], n, p=[0.45, 0.15, 0.40])
    purpose        = rng.choice(
        ["debt_consolidation", "credit_card", "home_improvement",
         "major_purchase", "medical", "other"],
        n, p=[0.40, 0.25, 0.10, 0.10, 0.05, 0.10]
    )
    emp_length = rng.choice(
        ["< 1 year", "1-3 years", "4-6 years", "7-9 years", "10+ years"],
        n, p=[0.10, 0.25, 0.25, 0.20, 0.20]
    )
    grade = rng.choice(list("ABCDEFG"), n, p=[0.20, 0.25, 0.22, 0.15, 0.10, 0.05, 0.03])

    # Default probability is driven by the underlying risk factors
    log_odds = (
        -4.5
        + (700 - fico) * 0.015
        + dti * 0.06
        + delinq_2yrs * 0.5
        + (int_rate - 10) * 0.12
        - np.log1p(annual_inc) * 0.15
        + (revol_util - 50) * 0.012
    )
    p_default = 1 / (1 + np.exp(-log_odds))
    default = rng.binomial(1, p_default)

    df = pd.DataFrame({
        "loan_amnt":      loan_amnt,
        "int_rate":       int_rate.round(2),
        "annual_inc":     annual_inc.round(2),
        "dti":            dti.round(2),
        "fico_score":     fico.round(0).astype(int),
        "delinq_2yrs":    delinq_2yrs,
        "open_acc":       open_acc,
        "revol_util":     revol_util.round(2),
        "total_acc":      total_acc,
        "home_ownership": home_ownership,
        "purpose":        purpose,
        "emp_length":     emp_length,
        "grade":          grade,
        "default":        default,
    })
    # Introduce ~3% missing values to make preprocessing realistic
    for col in ["int_rate", "dti", "revol_util", "emp_length"]:
        mask = rng.random(n) < 0.03
        df.loc[mask, col] = np.nan

    return df


df = generate_loan_dataset(n=10_000, random_state=RANDOM_STATE)
print(f"Dataset shape : {df.shape}")
print(f"Default rate  : {df['default'].mean():.2%}")
df.head()


## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

Before building any model we thoroughly explore the data to understand:
- **Class distribution** (imbalance severity)
- **Numeric feature distributions** split by default status
- **Categorical feature default rates**
- **Correlation structure** of numeric features

In [ ]:
# ── Basic info ────────────────────────────────────────────────────────────────
print("=== Dataset Info ===")
df.info()
print("\n=== Missing Values ===")
print(df.isnull().sum())
print("\n=== Target Distribution ===")
print(df['default'].value_counts())


In [ ]:
# ── 3.1  Class distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['default'].value_counts()
axes[0].bar(['No Default (0)', 'Default (1)'], counts.values,
            color=[PALETTE[0], PALETTE[3]])
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 50, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=10)

axes[1].pie(counts.values, labels=['No Default', 'Default'],
            autopct='%1.1f%%', colors=[PALETTE[0], PALETTE[3]],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('Class Proportion', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/01_class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Default rate: {df['default'].mean():.2%}")


In [ ]:
# ── 3.2  Numeric feature distributions (default vs non-default) ──────────────
NUM_COLS = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
            'fico_score', 'delinq_2yrs', 'revol_util', 'total_acc']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, col in enumerate(NUM_COLS):
    ax = axes[i]
    for label, color in [(0, PALETTE[0]), (1, PALETTE[3])]:
        data = df.loc[df['default'] == label, col].dropna()
        ax.hist(data, bins=30, alpha=0.6, color=color,
                label='Default' if label == 1 else 'No Default', density=True)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.suptitle('Numeric Feature Distributions by Default Status',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/02_numeric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.3  Default rate by categorical features ────────────────────────────────
CAT_COLS = ['home_ownership', 'purpose', 'emp_length', 'grade']

fig, axes = plt.subplots(2, 2, figsize=(18, 10))
axes = axes.flatten()

for i, col in enumerate(CAT_COLS):
    ax = axes[i]
    order = df.groupby(col)['default'].mean().sort_values(ascending=False).index
    default_rates = df.groupby(col)['default'].mean().reindex(order)
    bars = ax.bar(range(len(default_rates)), default_rates.values,
                  color=sns.color_palette("muted", len(default_rates)))
    ax.set_xticks(range(len(default_rates)))
    ax.set_xticklabels(default_rates.index, rotation=30, ha='right', fontsize=9)
    ax.set_title(f'Default Rate by {col.replace("_"," ").title()}',
                 fontweight='bold', fontsize=12)
    ax.set_ylabel('Default Rate')
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                f'{bar.get_height():.1%}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/03_categorical_default_rates.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.4  Correlation heatmap ─────────────────────────────────────────────────
num_df = df[NUM_COLS + ['default']].copy()
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=16, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/04_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 3.5  Box plots – key numeric features vs default ─────────────────────────
KEY_COLS = ['fico_score', 'int_rate', 'dti', 'annual_inc']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, col in enumerate(KEY_COLS):
    ax = axes[i]
    df.boxplot(column=col, by='default', ax=ax,
               boxprops=dict(color=PALETTE[0]),
               medianprops=dict(color='red', linewidth=2),
               whiskerprops=dict(color=PALETTE[0]),
               capprops=dict(color=PALETTE[0]),
               flierprops=dict(marker='o', alpha=0.3, markersize=3))
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Default Status')

plt.suptitle('Key Feature Distributions by Default Status', fontsize=14,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/05_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()


## 4. Data Preprocessing <a id='4'></a>

We build a **sklearn `ColumnTransformer` pipeline** that:

1. **Imputes** missing values (median for numerics, most-frequent for categoricals)
2. **Scales** numeric features with `StandardScaler`
3. **One-hot encodes** categorical features

The preprocessor is fitted **only on training data** and then applied to test data to prevent data leakage.

In [ ]:
# ── Train / test split (stratified) ──────────────────────────────────────────
TARGET = 'default'
FEATURES = [c for c in df.columns if c != TARGET]

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set : {X_train.shape[0]:,} samples  | Default rate: {y_train.mean():.2%}")
print(f"Test set     : {X_test.shape[0]:,} samples  | Default rate: {y_test.mean():.2%}")


In [ ]:
# ── Feature type lists ────────────────────────────────────────────────────────
NUMERIC_FEATURES = ['loan_amnt', 'int_rate', 'annual_inc', 'dti',
                    'fico_score', 'delinq_2yrs', 'open_acc',
                    'revol_util', 'total_acc']

CATEGORICAL_FEATURES = ['home_ownership', 'purpose', 'emp_length', 'grade']

# ── Pipelines for each feature type ──────────────────────────────────────────
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer,  NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES),
])

# ── Fit on training data only ─────────────────────────────────────────────────
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# Feature names after one-hot encoding
ohe_cats = (preprocessor
            .named_transformers_['cat']
            .named_steps['onehot']
            .get_feature_names_out(CATEGORICAL_FEATURES).tolist())
FEATURE_NAMES = NUMERIC_FEATURES + ohe_cats

print(f"Preprocessed training shape : {X_train_prep.shape}")
print(f"Preprocessed test shape     : {X_test_prep.shape}")
print(f"Total features after OHE    : {len(FEATURE_NAMES)}")


## 5. Class Imbalance – SMOTE <a id='5'></a>

Class imbalance is a common challenge in credit-risk modelling. We address it with
**SMOTE (Synthetic Minority Over-sampling Technique)**, which generates new synthetic
minority-class samples by interpolating between existing ones.

### Key rules:
- SMOTE is applied **only to the training set** – the test set remains untouched.
- We show a **before / after comparison** of class counts and a 2-D PCA visualisation.

In [ ]:
from sklearn.decomposition import PCA

# ── Before SMOTE ─────────────────────────────────────────────────────────────
before_counts = pd.Series(y_train).value_counts().sort_index()
print("Class distribution BEFORE SMOTE:")
print(before_counts)
print(f"  Imbalance ratio: 1 : {before_counts[0]/before_counts[1]:.1f}")

# ── Apply SMOTE ───────────────────────────────────────────────────────────────
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train_prep, y_train)

after_counts = pd.Series(y_train_bal).value_counts().sort_index()
print("\nClass distribution AFTER SMOTE:")
print(after_counts)
print(f"  Imbalance ratio: 1 : {after_counts[0]/after_counts[1]:.1f}")


In [ ]:
# ── Before / after bar chart ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for ax, counts, title in zip(
    axes,
    [before_counts, after_counts],
    ['Before SMOTE', 'After SMOTE']
):
    bars = ax.bar(['No Default (0)', 'Default (1)'], counts.values,
                  color=[PALETTE[0], PALETTE[3]], edgecolor='white', linewidth=1.2)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Sample Count')
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                f'{int(bar.get_height()):,}', ha='center', fontsize=11)

plt.suptitle('Class Imbalance: Before vs After SMOTE', fontsize=16,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/06_smote_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2-D PCA visualisation ─────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for ax, X_vis, y_vis, title in [
    (axes[0], X_train_prep, y_train, 'Before SMOTE'),
    (axes[1], X_train_bal,  y_train_bal, 'After SMOTE'),
]:
    X_2d = pca.fit_transform(X_vis)
    for cls, label, color in [(0, 'No Default', PALETTE[0]), (1, 'Default', PALETTE[3])]:
        mask = y_vis == cls
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
                   c=[color], label=label, alpha=0.4, s=10)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.legend()
    ax.set_xlabel('PC 1')
    ax.set_ylabel('PC 2')

plt.suptitle('PCA 2-D Projection – SMOTE Effect', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/07_pca_smote.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Model Training & Comparison <a id='6'></a>

We train **five classifiers** on the SMOTE-balanced training set and evaluate them on the
hold-out test set using a comprehensive set of metrics:

| Metric | Why it matters |
|--------|---------------|
| Accuracy | Overall correctness |
| Precision | How many flagged defaults are real defaults |
| Recall | How many actual defaults we catch |
| F1 Score | Harmonic mean of Precision & Recall |
| ROC-AUC | Ranking ability across all thresholds |
| PR-AUC | Precision-Recall trade-off (important for imbalanced data) |

In [ ]:
def evaluate_model(model, X_tr, y_tr, X_te, y_te, name):
    """Fit model and return a dict of evaluation metrics."""
    model.fit(X_tr, y_tr)
    y_pred  = model.predict(X_te)
    y_proba = model.predict_proba(X_te)[:, 1]

    return {
        'Model':     name,
        'Accuracy':  accuracy_score(y_te, y_pred),
        'Precision': precision_score(y_te, y_pred, zero_division=0),
        'Recall':    recall_score(y_te, y_pred, zero_division=0),
        'F1':        f1_score(y_te, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_te, y_proba),
        'PR-AUC':    average_precision_score(y_te, y_proba),
        '_model':    model,
        '_proba':    y_proba,
        '_pred':     y_pred,
    }


In [ ]:
# ── Define models ─────────────────────────────────────────────────────────────
base_models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=200, random_state=RANDOM_STATE
    ),
}

if XGBOOST_AVAILABLE:
    base_models['XGBoost'] = XGBClassifier(
        n_estimators=200, random_state=RANDOM_STATE,
        eval_metric='logloss', verbosity=0
    )

if LIGHTGBM_AVAILABLE:
    base_models['LightGBM'] = LGBMClassifier(
        n_estimators=200, random_state=RANDOM_STATE,
        class_weight='balanced', verbosity=-1
    )

# ── Train & evaluate ──────────────────────────────────────────────────────────
results = []
fitted_models = {}

for name, model in base_models.items():
    print(f"Training {name}...", end=' ')
    res = evaluate_model(model, X_train_bal, y_train_bal,
                         X_test_prep, y_test, name)
    fitted_models[name] = res.pop('_model')
    probas = {name: res.pop('_proba')}
    preds  = {name: res.pop('_pred')}
    results.append(res)
    # Store probas/preds for later use
    fitted_models[name + '_proba'] = list(probas.values())[0]
    fitted_models[name + '_pred']  = list(preds.values())[0]
    print(f"✓  ROC-AUC={res['ROC-AUC']:.4f}  PR-AUC={res['PR-AUC']:.4f}")

results_df = pd.DataFrame(results).set_index('Model').round(4)
print("\n=== Model Comparison ===")
print(results_df.to_string())


In [ ]:
# ── Metrics bar chart ─────────────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC', 'PR-AUC']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
colors = sns.color_palette("muted", len(results_df))

for i, metric in enumerate(metrics):
    ax = axes[i]
    bars = ax.bar(range(len(results_df)), results_df[metric].values,
                  color=colors, edgecolor='white', linewidth=0.8)
    ax.set_xticks(range(len(results_df)))
    ax.set_xticklabels(results_df.index, rotation=20, ha='right', fontsize=9)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison – All Metrics', fontsize=17, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/08_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for name in base_models:
    proba = fitted_models[name + '_proba']
    # ROC
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    axes[0].plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})', linewidth=2)
    # PR
    prec, rec, _ = precision_recall_curve(y_test, proba)
    pr_auc = average_precision_score(y_test, proba)
    axes[1].plot(rec, prec, label=f'{name} (AP={pr_auc:.3f})', linewidth=2)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random')
axes[0].set_title('ROC Curves', fontsize=14, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(fontsize=9)

axes[1].axhline(y=y_test.mean(), color='k', linestyle='--', alpha=0.5, label='Baseline')
axes[1].set_title('Precision-Recall Curves', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/09_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Hyperparameter Tuning <a id='7'></a>

Based on the baseline results we tune the **top 2–3 models** using `GridSearchCV` with
5-fold stratified cross-validation, optimising for **PR-AUC** which is more sensitive to
the minority class than ROC-AUC in imbalanced settings.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ─── 7.1  Random Forest ───────────────────────────────────────────────────────
print("Tuning Random Forest ...")
rf_param_grid = {
    'n_estimators':      [100, 200],
    'max_depth':         [None, 10, 20],
    'min_samples_leaf':  [1, 5],
    'max_features':      ['sqrt', 'log2'],
}
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1),
    rf_param_grid, cv=CV, scoring='average_precision',
    n_jobs=-1, verbose=0, refit=True
)
rf_grid.fit(X_train_bal, y_train_bal)
print(f"  Best params : {rf_grid.best_params_}")
print(f"  CV PR-AUC   : {rf_grid.best_score_:.4f}")


In [ ]:
# ─── 7.2  Gradient Boosting ───────────────────────────────────────────────────
print("Tuning Gradient Boosting ...")
gb_param_grid = {
    'n_estimators':  [100, 200],
    'learning_rate': [0.05, 0.10, 0.20],
    'max_depth':     [3, 5],
    'subsample':     [0.8, 1.0],
}
gb_grid = GridSearchCV(
    GradientBoostingClassifier(random_state=RANDOM_STATE),
    gb_param_grid, cv=CV, scoring='average_precision',
    n_jobs=-1, verbose=0, refit=True
)
gb_grid.fit(X_train_bal, y_train_bal)
print(f"  Best params : {gb_grid.best_params_}")
print(f"  CV PR-AUC   : {gb_grid.best_score_:.4f}")


In [ ]:
# ─── 7.3  XGBoost (if available) ─────────────────────────────────────────────
if XGBOOST_AVAILABLE:
    print("Tuning XGBoost ...")
    xgb_param_grid = {
        'n_estimators':  [100, 200],
        'learning_rate': [0.05, 0.10],
        'max_depth':     [3, 6],
        'subsample':     [0.8, 1.0],
        'colsample_bytree': [0.8, 1.0],
    }
    xgb_grid = GridSearchCV(
        XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss', verbosity=0),
        xgb_param_grid, cv=CV, scoring='average_precision',
        n_jobs=-1, verbose=0, refit=True
    )
    xgb_grid.fit(X_train_bal, y_train_bal)
    print(f"  Best params : {xgb_grid.best_params_}")
    print(f"  CV PR-AUC   : {xgb_grid.best_score_:.4f}")
else:
    print("XGBoost not available – skipping tuning.")


In [ ]:
# ── Collect tuned results ─────────────────────────────────────────────────────
tuned_models = {
    'RF (Tuned)': rf_grid.best_estimator_,
    'GB (Tuned)': gb_grid.best_estimator_,
}
if XGBOOST_AVAILABLE:
    tuned_models['XGB (Tuned)'] = xgb_grid.best_estimator_

tuned_results = []
for name, model in tuned_models.items():
    y_pred  = model.predict(X_test_prep)
    y_proba = model.predict_proba(X_test_prep)[:, 1]
    tuned_results.append({
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'ROC-AUC':   roc_auc_score(y_test, y_proba),
        'PR-AUC':    average_precision_score(y_test, y_proba),
    })
    # Update fitted_models store
    fitted_models[name] = model
    fitted_models[name + '_proba'] = y_proba
    fitted_models[name + '_pred']  = y_pred

tuned_df = pd.DataFrame(tuned_results).set_index('Model').round(4)

# Combine with baseline
all_results = pd.concat([results_df, tuned_df]).sort_values('PR-AUC', ascending=False)
print("=== All Models (Baseline + Tuned) ranked by PR-AUC ===")
print(all_results.to_string())


In [ ]:
# ── Visual comparison: baseline vs tuned ─────────────────────────────────────
for_plot = all_results[['F1', 'ROC-AUC', 'PR-AUC']].copy()

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(for_plot))
width = 0.25
bars1 = ax.bar(x - width, for_plot['F1'],      width, label='F1',      color=PALETTE[0])
bars2 = ax.bar(x,          for_plot['ROC-AUC'], width, label='ROC-AUC', color=PALETTE[1])
bars3 = ax.bar(x + width,  for_plot['PR-AUC'],  width, label='PR-AUC',  color=PALETTE[2])

ax.set_xticks(x)
ax.set_xticklabels(for_plot.index, rotation=25, ha='right', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title('Baseline vs Tuned Models – Key Metrics', fontsize=15, fontweight='bold')
ax.legend()
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/10_tuned_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 8. Model Evaluation <a id='8'></a>

Detailed evaluation of the **best model** selected by PR-AUC, including:
- Classification report
- Confusion matrix
- Learning curves

In [ ]:
# ── Identify best model ───────────────────────────────────────────────────────
BEST_NAME = all_results['PR-AUC'].idxmax()
best_model = fitted_models[BEST_NAME]
best_proba = fitted_models[BEST_NAME + '_proba']
best_pred  = fitted_models[BEST_NAME + '_pred']

print(f"Best model: {BEST_NAME}")
print(f"  ROC-AUC : {roc_auc_score(y_test, best_proba):.4f}")
print(f"  PR-AUC  : {average_precision_score(y_test, best_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['No Default', 'Default']))


In [ ]:
# ── Confusion matrix ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

cm = confusion_matrix(y_test, best_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=['No Default', 'Default'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title(f'Confusion Matrix – {BEST_NAME}\n(default threshold 0.5)',
                  fontsize=12, fontweight='bold')

# Normalised
cm_norm = confusion_matrix(y_test, best_pred, normalize='true')
disp2 = ConfusionMatrixDisplay(confusion_matrix=cm_norm,
                                display_labels=['No Default', 'Default'])
disp2.plot(ax=axes[1], cmap='Blues', colorbar=False,
           values_format='.2%')
axes[1].set_title(f'Normalised Confusion Matrix – {BEST_NAME}',
                  fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/11_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Learning curves ───────────────────────────────────────────────────────────
train_sizes, train_scores, val_scores = learning_curve(
    best_model, X_train_bal, y_train_bal,
    cv=CV, scoring='average_precision',
    train_sizes=np.linspace(0.1, 1.0, 8), n_jobs=-1
)

train_mean = train_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_mean   = val_scores.mean(axis=1)
val_std    = val_scores.std(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_sizes, train_mean, 'o-', color=PALETTE[0], label='Training PR-AUC')
ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std,
                alpha=0.15, color=PALETTE[0])
ax.plot(train_sizes, val_mean, 'o-', color=PALETTE[3], label='Cross-Val PR-AUC')
ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std,
                alpha=0.15, color=PALETTE[3])
ax.set_title(f'Learning Curves – {BEST_NAME}', fontsize=14, fontweight='bold')
ax.set_xlabel('Training Set Size')
ax.set_ylabel('PR-AUC')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/12_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Model Explainability (SHAP) <a id='9'></a>

**SHAP (SHapley Additive exPlanations)** assigns each feature a contribution value for
each prediction, providing both global and local interpretability.

- **Summary plot** – global feature importance (impact on model output)
- **Force plot** – single-prediction explanation (local interpretability)

In [ ]:
if SHAP_AVAILABLE:
    # Use a tree explainer for tree-based models, else use KernelExplainer
    model_for_shap = best_model

    try:
        explainer = shap.TreeExplainer(model_for_shap)
        shap_values = explainer.shap_values(X_test_prep)
        # For binary classifiers TreeExplainer returns list [class0, class1]
        if isinstance(shap_values, list):
            sv = shap_values[1]
        else:
            sv = shap_values
    except Exception:
        print("TreeExplainer failed – falling back to LinearExplainer / KernelExplainer")
        background = shap.kmeans(X_train_bal, 50)
        explainer  = shap.KernelExplainer(model_for_shap.predict_proba, background)
        shap_values_raw = explainer.shap_values(X_test_prep[:100])
        sv = shap_values_raw[1] if isinstance(shap_values_raw, list) else shap_values_raw

    # ── Summary plot ─────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 7))
    shap.summary_plot(sv, X_test_prep,
                      feature_names=FEATURE_NAMES,
                      show=False, plot_size=None)
    plt.title(f'SHAP Summary Plot – {BEST_NAME}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/13_shap_summary.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("SHAP not available. Install with: pip install shap")


In [ ]:
if SHAP_AVAILABLE:
    # ── Bar plot (mean |SHAP|) ─────────────────────────────────────────────
    shap_importance = pd.DataFrame({
        'Feature': FEATURE_NAMES,
        'Mean |SHAP|': np.abs(sv).mean(axis=0)
    }).sort_values('Mean |SHAP|', ascending=False).head(15)

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(shap_importance['Feature'][::-1],
                   shap_importance['Mean |SHAP|'][::-1],
                   color=sns.color_palette("muted", 15))
    ax.set_title(f'Top 15 Features by Mean |SHAP| – {BEST_NAME}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Mean |SHAP value|')
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/14_shap_bar.png', dpi=150, bbox_inches='tight')
    plt.show()


In [ ]:
if SHAP_AVAILABLE:
    # ── Force plot for a single high-risk prediction ───────────────────────
    # Find the sample with the highest predicted default probability
    idx = np.argmax(fitted_models[BEST_NAME + '_proba'])
    print(f"Sample index with highest default probability: {idx}")
    print(f"Predicted probability: {fitted_models[BEST_NAME + '_proba'][idx]:.4f}")
    print(f"Actual label: {y_test.iloc[idx]}")

    shap.initjs()
    force = shap.force_plot(
        explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray))
        else explainer.expected_value,
        sv[idx],
        X_test_prep[idx],
        feature_names=FEATURE_NAMES,
        matplotlib=True,
        show=False
    )
    plt.savefig(f'{PLOTS_DIR}/15_shap_force_plot.png', dpi=150, bbox_inches='tight')
    plt.show()


## 10. Business-Oriented Evaluation <a id='10'></a>

### Cost Matrix

In loan default prediction, misclassification costs are **asymmetric**:

| | Predicted: No Default | Predicted: Default |
|---|---|---|
| **Actual: No Default** | ✅ True Negative (0) | ❌ False Positive (cost = 1) |
| **Actual: Default** | ❌ False Negative (cost = 5) | ✅ True Positive (0) |

The cost of a **false negative** (missing a real defaulter) is **5× higher** than a
false positive (incorrectly flagging a good customer), reflecting the financial loss
from an undetected default.

We find the **optimal threshold** that minimises expected business cost.

In [ ]:
# ── Business cost matrix ──────────────────────────────────────────────────────
COST_FP = 1   # Cost of false positive  (good customer denied)
COST_FN = 5   # Cost of false negative  (defaulter not caught)

thresholds = np.linspace(0.01, 0.99, 200)
costs = []

for thresh in thresholds:
    y_pred_t = (best_proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    total_cost = fp * COST_FP + fn * COST_FN
    costs.append(total_cost)

optimal_threshold = thresholds[np.argmin(costs)]
print(f"Optimal business threshold: {optimal_threshold:.3f}")
print(f"Minimum business cost:      {min(costs):,}")

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(thresholds, costs, color=PALETTE[0], linewidth=2)
ax.axvline(optimal_threshold, color=PALETTE[3], linestyle='--', linewidth=2,
           label=f'Optimal threshold = {optimal_threshold:.3f}')
ax.set_title(f'Business Cost vs Threshold – {BEST_NAME}\n'
             f'(FP cost={COST_FP}, FN cost={COST_FN})',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Decision Threshold')
ax.set_ylabel('Total Business Cost')
ax.legend()
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/16_business_cost_threshold.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Confusion matrix at optimal threshold ─────────────────────────────────────
y_pred_opt = (best_proba >= optimal_threshold).astype(int)

print("=== Performance at Optimal Business Threshold ===")
print(classification_report(y_test, y_pred_opt, target_names=['No Default', 'Default']))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, norm, title in [
    (axes[0], None,   f'Confusion Matrix at threshold={optimal_threshold:.3f}'),
    (axes[1], 'true', f'Normalised – threshold={optimal_threshold:.3f}'),
]:
    cm_opt = confusion_matrix(y_test, y_pred_opt, normalize=norm)
    fmt = '.2%' if norm else 'd'
    disp = ConfusionMatrixDisplay(cm_opt, display_labels=['No Default', 'Default'])
    disp.plot(ax=ax, cmap='Oranges', colorbar=False, values_format=fmt)
    ax.set_title(title, fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/17_confusion_matrix_optimal.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Calibration curve ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

for name in list(base_models.keys())[:3]:
    prob = fitted_models[name + '_proba']
    CalibrationDisplay.from_predictions(
        y_test, prob, n_bins=10, ax=ax, name=name
    )

ax.set_title('Calibration Curves (Reliability Diagram)', fontsize=14, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/18_calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Feature importance (native) ───────────────────────────────────────────────
if hasattr(best_model, 'feature_importances_'):
    fi = pd.Series(best_model.feature_importances_, index=FEATURE_NAMES)
    fi_top = fi.nlargest(15).sort_values()

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(fi_top.index, fi_top.values,
                   color=sns.color_palette("muted", len(fi_top)))
    ax.set_title(f'Top 15 Feature Importances – {BEST_NAME}',
                 fontsize=14, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/19_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("Model does not expose feature_importances_; use SHAP instead.")


## 11. Final Model Selection & Export <a id='11'></a>

We save the **best model** and the **preprocessor** as separate `joblib` files so they
can be loaded independently in a serving environment.

In [ ]:
MODEL_PATH       = 'best_model.joblib'
PREPROCESSOR_PATH = 'preprocessor.joblib'
THRESHOLD_PATH   = 'optimal_threshold.json'

joblib.dump(best_model,  MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

with open(THRESHOLD_PATH, 'w') as f:
    json.dump({'optimal_threshold': float(optimal_threshold),
               'model_name': BEST_NAME,
               'cost_fp': COST_FP, 'cost_fn': COST_FN}, f, indent=2)

print(f"✅ Model saved       → {MODEL_PATH}")
print(f"✅ Preprocessor saved → {PREPROCESSOR_PATH}")
print(f"✅ Threshold saved   → {THRESHOLD_PATH}")

# ── Verify round-trip ─────────────────────────────────────────────────────────
loaded_model = joblib.load(MODEL_PATH)
loaded_prep  = joblib.load(PREPROCESSOR_PATH)
sample_pred  = loaded_model.predict_proba(
                   loaded_prep.transform(X_test.iloc[:5])
               )[:, 1]
print(f"\nRound-trip check – first 5 probabilities: {sample_pred.round(4)}")


## 12. Prediction Function <a id='12'></a>

A simple, reusable function that accepts a raw dictionary (or DataFrame row) and
returns the **default probability** and **decision** at the optimal threshold.

In [ ]:
def predict_default_risk(
    applicant_data: dict,
    preprocessor_path: str = PREPROCESSOR_PATH,
    model_path: str = MODEL_PATH,
    threshold_path: str = THRESHOLD_PATH,
) -> dict:
    """
    Predict default probability for a new loan applicant.

    Parameters
    ----------
    applicant_data : dict
        Raw applicant features (same columns as training data).
    preprocessor_path : str
        Path to the saved ColumnTransformer.
    model_path : str
        Path to the saved classifier.
    threshold_path : str
        Path to the JSON file containing the optimal threshold.

    Returns
    -------
    dict with keys:
        default_probability : float
        decision            : str  ('DEFAULT RISK' | 'APPROVE')
        threshold_used      : float
    """
    prep  = joblib.load(preprocessor_path)
    model = joblib.load(model_path)
    with open(threshold_path) as f:
        threshold = json.load(f)['optimal_threshold']

    row   = pd.DataFrame([applicant_data])
    X_raw = prep.transform(row)
    prob  = model.predict_proba(X_raw)[0, 1]

    return {
        'default_probability': round(float(prob), 4),
        'decision':            'DEFAULT RISK' if prob >= threshold else 'APPROVE',
        'threshold_used':      round(float(threshold), 4),
    }


# ── Demo ──────────────────────────────────────────────────────────────────────
sample_applicant = {
    'loan_amnt':      15_000,
    'int_rate':       14.5,
    'annual_inc':     55_000,
    'dti':            18.0,
    'fico_score':     660,
    'delinq_2yrs':    1,
    'open_acc':       8,
    'revol_util':     65.0,
    'total_acc':      22,
    'home_ownership': 'RENT',
    'purpose':        'debt_consolidation',
    'emp_length':     '4-6 years',
    'grade':          'C',
}

result = predict_default_risk(sample_applicant)
print("=== Prediction Result ===")
for k, v in result.items():
    print(f"  {k:<25}: {v}")


## 13. Conclusion & Business Recommendations <a id='13'></a>

### 📊 Model Performance Summary

| Aspect | Finding |
|--------|---------|
| Best model | Selected by highest PR-AUC on hold-out test set |
| Class imbalance | Addressed with SMOTE on training data only |
| Hyperparameter tuning | GridSearchCV 5-fold CV, optimising PR-AUC |
| Optimal threshold | Derived from cost-sensitive business cost matrix |

---

### 💡 Business Recommendations

1. **Deploy at the cost-optimised threshold** – the default 0.5 cut-off ignores the
   asymmetric costs of false negatives (missed defaults) vs false positives (good
   customers declined). The optimised threshold saves significant expected losses.

2. **Monitor drift regularly** – credit risk models degrade over economic cycles.
   Schedule monthly PSI (Population Stability Index) checks on input features and
   model scores.

3. **Segment the portfolio** – different customer segments (e.g., first-time borrowers
   vs. established customers) may require separate models or adjusted thresholds.

4. **Incorporate macroeconomic signals** – variables like unemployment rate, interest
   rate environment, and GDP growth improve through-the-cycle performance.

5. **Re-calibrate the model annually** – use Platt scaling or isotonic regression to
   keep predicted probabilities aligned with observed default rates as the portfolio mix
   evolves.

6. **Fair lending compliance** – audit the model for disparate impact across protected
   attributes (race, gender, age) before production deployment.

7. **Explainability for regulators** – SHAP explanations enable adverse-action notices
   mandated by fair lending laws (e.g., ECOA, GDPR Article 22).

---

### 🗂️ Artefacts Produced

| File | Description |
|------|-------------|
| `best_model.joblib` | Serialised trained classifier |
| `preprocessor.joblib` | Fitted ColumnTransformer |
| `optimal_threshold.json` | Business cost–optimised decision threshold |
| `plots/*.png` | All visualisations |
| `requirements.txt` | Environment specification |

## 14. Requirements <a id='14'></a>

Generate a `requirements.txt` pinned to the currently installed versions.

In [ ]:
import subprocess, sys

pkgs = [
    'numpy', 'pandas', 'scikit-learn', 'imbalanced-learn',
    'xgboost', 'lightgbm', 'shap', 'matplotlib', 'seaborn',
    'joblib', 'optuna',
]

lines = []
for pkg in pkgs:
    try:
        result = subprocess.run(
            [sys.executable, '-m', 'pip', 'show', pkg],
            capture_output=True, text=True
        )
        version_line = [l for l in result.stdout.splitlines() if l.startswith('Version')]
        if version_line:
            version = version_line[0].split(': ')[1].strip()
            lines.append(f'{pkg}=={version}')
        else:
            lines.append(f'# {pkg} not installed')
    except Exception:
        lines.append(f'# {pkg} – version check failed')

req_content = '\n'.join(lines) + '\n'
with open('requirements.txt', 'w') as f:
    f.write(req_content)

print("=== requirements.txt ===")
print(req_content)
